## 0. Package loading and installation



In [10]:
# Commented out IPython magic to ensure Python compatibility.
# For Jupyter/Colab notebooks
%reset -f
import gc
gc.collect()

import numpy as np
import pandas as pd
import time

#conda activate surv-deephit
#conda install ipykernel -y
#conda install -c conda-forge pytorch torchtuples pycox
#conda install pytorch torchvision torchaudio pytorch-cuda=11.8 -c pytorch -c nvidia
# conda install pycox torchtuples scikit-learn scikit-survival lifelines shap seaborn matplotlib scipy pandas -c conda-forge -y
# por si: conda install pycox torchtuples -c conda-forge -y

#Conda te avisa que va a hacer dos cambios porque estás instalando PyTorch con CUDA:
#conda-forge::cuda-cudart 12.9  →  nvidia::cuda-cudart 11.8

#Packages stored in : 
#conda env export --no-builds > "G:\My Drive\Alvacast\SISTRAT 2023\dh\environment.yml"

#Load packages in:
#conda activate base
#conda-lock install \
#  -n surv-deephit \
#  "G:\My Drive\Alvacast\SISTRAT 2023\dh\conda-lock.yml"

import sys
import subprocess

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Compute Device: {device}")

from sksurv.metrics import (
    concordance_index_ipcw,
    brier_score,
    integrated_brier_score
)
from sksurv.util import Surv



#Glimpse function
def glimpse(df, max_width=80):
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    for col in df.columns:
        dtype = df[col].dtype
        preview = df[col].astype(str).head(5).tolist()
        preview_str = ", ".join(preview)
        if len(preview_str) > max_width:
            preview_str = preview_str[:max_width] + "..."
        print(f"{col:<30} {str(dtype):<15} {preview_str}")
#Tabyl function
def tabyl(series):
    counts = series.value_counts(dropna=False)
    props = series.value_counts(normalize=True, dropna=False)
    return pd.DataFrame({"value": counts.index,
                         "n": counts.values,
                         "percent": props.values})
#clean_names
import re

def clean_names(df):
    """
    Mimic janitor::clean_names for pandas DataFrames.
    - Lowercase
    - Replace spaces and special chars with underscores
    - Remove non-alphanumeric/underscore
    """
    new_cols = []
    for col in df.columns:
        # lowercase
        col = col.lower()
        # replace spaces and special chars with underscore
        col = re.sub(r"[^\w]+", "_", col)
        # strip leading/trailing underscores
        col = col.strip("_")
        new_cols.append(col)
    df.columns = new_cols
    return df


✅ Compute Device: cuda


In [11]:
packages = ["torch", "torchtuples", "pycox"]

for p in packages:
    try:
        mod = __import__(p)
        print(f"✅ {p} installed | version:", getattr(mod, "__version__", "unknown"))
    except ImportError:
        print(f"❌ {p} NOT installed")


✅ torch installed | version: 2.5.1
✅ torchtuples installed | version: 0.2.2
✅ pycox installed | version: 0.3.0


## Load data

In [12]:

from pathlib import Path

BASE_DIR = Path(
    r"G:\My Drive\Alvacast\SISTRAT 2023\data\20241015_out\pred1"
)


import pickle

with open(BASE_DIR / "imputations_list_jan26.pkl", "rb") as f:
    imputations_list_jan26 = pickle.load(f)


imputation_nodum_1 = pd.read_parquet(
    BASE_DIR / "imputation_nondum_1.parquet",
    engine="fastparquet"
)

X_reduced_imp0 = pd.read_parquet(
    BASE_DIR / "X_reduced_imp0.parquet",
    engine="fastparquet"
)

imputation_1 = pd.read_parquet(
    BASE_DIR / "imputation_1.parquet",
    engine="fastparquet"
)


# Quick check
glimpse(imputation_nodum_1)
glimpse(imputation_1)
glimpse(X_reduced_imp0)

Rows: 88504 | Columns: 43
readmit_time_from_adm_m        float64         84.93548387096774, 12.833333333333334, 13.733333333333333, 11.966666666666667, 1...
death_time_from_adm_m          float64         84.93548387096774, 87.16129032258064, 117.2258064516129, 98.93548387096774, 37.9...
adm_age_rec3                   float64         31.53, 20.61, 42.52, 60.61, 45.08
porc_pobr                      float64         0.175679117441177, 0.187835901975632, 0.130412444472313, 0.133759185671806, 0.08...
dit_m                          float64         15.967741935483872, 5.833333333333334, 0.4752688172043005, 6.966666666666667, 6....
sex_rec                        object          man, man, man, woman, man
tenure_status_household        object          stays temporarily with a relative, owner/transferred dwellings/pays dividends, s...
cohabitation                   object          alone, family of origin, with couple/children, with couple/children, family of o...
sub_dep_icd10_status           obj

Load in python

In [15]:
if isinstance(imputations_list_jan26, list) and len(imputations_list_jan26) > 0:
    print("First element type:", type(imputations_list_jan26[0]))
    if isinstance(imputations_list_jan26[0], dict):
        print("First element keys:", imputations_list_jan26[0].keys())
    elif isinstance(imputations_list_jan26[0], (pd.DataFrame, np.ndarray)):
        print("First element shape:", imputations_list_jan26[0].shape)


First element type: <class 'pandas.core.frame.DataFrame'>
First element shape: (88504, 56)


This code block:

1.  **Imports the `pickle` library**: This library implements binary protocols for serializing and de-serializing a Python object structure.
2.  **Specifies the `file_path`**: It points to the `.pkl` file you selected.
3.  **Opens the file in binary read mode (`'rb'`)**: This is necessary for loading pickle files.
4.  **Loads the object**: `pickle.load(f)` reads the serialized object from the file and reconstructs it in memory.
5.  **Prints confirmation and basic information**: It verifies that the file was loaded and shows the type of the loaded object, and some details about the first element if it's a list containing common data structures.



#### Compare databases (transformed and original)

Inspect and compare the column names of two datasets: the first imputation from imputations_list_jan26 (which likely contains dummy variables) and imputation_nodum_1 (which, as its name suggests, probably doesn't have dummy variables).

In [16]:
# Inspect columns of the first imputation
cols_first_imp = imputations_list_jan26[0].columns.tolist()
print("First imputation columns:", cols_first_imp[:10], "... total:", len(cols_first_imp))

# Inspect columns of imputation_no_dum
cols_nodum = imputation_nodum_1.columns.tolist()
print("No-dum columns:", cols_nodum[:10], "... total:", len(cols_nodum))

# Compare overlap
common_cols = set(cols_first_imp).intersection(cols_nodum)
missing_in_imp = [c for c in cols_nodum if c not in cols_first_imp]
missing_in_nodum = [c for c in cols_first_imp if c not in cols_nodum]

print("Common columns:", len(common_cols))
print("Missing in imputations_list_jan26:", missing_in_imp)

# Inspect columns of the first imputation
cols_first_imp_raw = imputation_1.columns.tolist()
print("First imputation columns:", cols_first_imp_raw[:10], "... total:", len(cols_first_imp_raw))

# Compare overlap
common_cols_raw = set(cols_first_imp_raw).intersection(cols_nodum)
missing_in_imp_raw = [c for c in cols_nodum if c not in cols_first_imp_raw]

print("Common columns:", len(common_cols_raw))
print("Missing in imputations_list_jan26:", missing_in_imp_raw)
print(common_cols_raw)

import pandas as pd

# Example: choose a combination of variables that uniquely identify rows
key_vars = ["adm_age_rec3", "porc_pobr", "dit_m"]

# Take one imputation (first element of the list) and merge with the no-dum dataset
df_imp = imputations_list_jan26[0]
df_nodum = imputation_nodum_1

merged_check = pd.merge(
    df_imp[key_vars],
    df_nodum[key_vars],
    on=key_vars,
    how="inner"
)

print(f"Merged rows: {merged_check.shape[0]}")
print("Preview of merged check:")
print(merged_check.head())

#drop merge
del merged_check

import pandas as pd

# Example: choose a combination of variables that uniquely identify rows
key_vars_raw = ['dit_m',
            'readmit_time_from_adm_m',
            'death_time_from_adm_m',
            'adm_age_rec3']
# Take one imputation (first element of the list) and merge with the no-dum dataset
df_raw = imputation_1

merged_check_raw = pd.merge(
    df_imp[key_vars],
    df_raw[key_vars],
    on=key_vars,
    how="inner"
)

print(f"Merged rows: {merged_check_raw.shape[0]}")
print("Preview of merged check:")
print(merged_check_raw.head())
print(f"{(merged_check_raw.shape[0] / imputation_1.shape[0] * 100):.2f}%")
#drop merge
del merged_check_raw


First imputation columns: ['adm_age_rec3', 'porc_pobr', 'dit_m', 'tenure_status_household', 'prim_sub_freq_rec', 'national_foreign', 'urbanicity_cat', 'ed_attainment_corr', 'evaluacindelprocesoteraputico', 'eva_consumo'] ... total: 56
No-dum columns: ['readmit_time_from_adm_m', 'death_time_from_adm_m', 'adm_age_rec3', 'porc_pobr', 'dit_m', 'sex_rec', 'tenure_status_household', 'cohabitation', 'sub_dep_icd10_status', 'any_violence'] ... total: 43
Common columns: 24
Missing in imputations_list_jan26: ['readmit_time_from_adm_m', 'death_time_from_adm_m', 'sex_rec', 'cohabitation', 'sub_dep_icd10_status', 'any_violence', 'tr_outcome', 'adm_motive', 'first_sub_used', 'primary_sub_mod', 'tipo_de_vivienda_rec2', 'plan_type_corr', 'occupation_condition_corr24', 'marital_status_rec', 'readmit_event', 'death_event', 'readmit_time_from_disch_m', 'death_time_from_disch_m', 'center_id']
First imputation columns: ['readmit_time_from_adm_m', 'death_time_from_adm_m', 'adm_age_rec3', 'porc_pobr', 'dit_m

### Create bins for followup (landmarks)

This code prepares your data for survival analysis. It extracts the time until an event (like readmission or death) and whether that event actually happened for each patient from the df_nodum dataset. Then, it automatically creates a set of important time points, called an 'evaluation grid', which are specific moments to assess the model's performance on both readmission and death outcomes.


In [17]:
import numpy as np

# Required columns for survival outcomes
required = ["readmit_time_from_disch_m", "readmit_event",
            "death_time_from_disch_m", "death_event"]

# Check that df_raw has all required columns
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise KeyError(f"df_nodum is missing columns: {missing}")

# Create time/event arrays directly from df_raw
time_readm = df_raw["readmit_time_from_adm_m"].to_numpy()
event_readm = (df_raw["readmit_event"].to_numpy() == 1)

time_death = df_raw["death_time_from_adm_m"].to_numpy()
event_death = (df_nodum["death_event"].to_numpy() == 1)

print("Arrays created for df_raw:")
print("Readmission times:", time_readm[:5])
print("Readmission events:", event_readm[:5])
print("Death times:", time_death[:5])
print("Death events:", event_death[:5])

# Build evaluation grids (quantiles of event times)
event_times_readm = time_readm[event_readm]
event_times_death = time_death[event_death]

if len(event_times_readm) < 5 or len(event_times_death) < 5:
    raise ValueError("Too few events in df_raw to build reliable time grids.")

times_eval_readm = np.unique(np.quantile(event_times_readm, np.linspace(0.05, 0.95, 50)))
times_eval_death = np.unique(np.quantile(event_times_death, np.linspace(0.05, 0.95, 50)))

print("Eval times (readmission):", times_eval_readm[:5], "...", times_eval_readm[-5:])
print("Eval times (death):", times_eval_death[:5], "...", times_eval_death[-5:])


Arrays created for df_raw:
Readmission times: [84.93548387 12.83333333 13.73333333 11.96666667 14.25806452]
Readmission events: [False  True  True  True  True]
Death times: [ 84.93548387  87.16129032 117.22580645  98.93548387  37.93548387]
Death events: [False False False False False]
Eval times (readmission): [3.93548387 4.77419355 5.45058701 6.06492649 6.67741935] ... [54.44173469 58.41566162 63.23333333 68.54767171 74.68983871]
Eval times (death): [4.16290323 5.43022383 6.68564845 8.24254115 9.77961817] ... [81.92700461 85.41186103 88.78518762 93.5538183  99.21935484]


Prepare survival data



In [18]:
import numpy as np

# Step 1. Extract survival outcomes directly from df_raw
time_readm = df_raw["readmit_time_from_adm_m"].to_numpy()
event_readm = (df_raw["readmit_event"].to_numpy() == 1)

time_death = df_raw["death_time_from_adm_m"].to_numpy()
event_death = (df_raw["death_event"].to_numpy() == 1)

# Step 2. Build structured arrays (Surv objects)
y_surv_readm = np.empty(len(time_readm), dtype=[("event", "?"), ("time", "<f8")])
y_surv_readm["event"] = event_readm
y_surv_readm["time"] = time_readm

y_surv_death = np.empty(len(time_death), dtype=[("event", "?"), ("time", "<f8")])
y_surv_death["event"] = event_death
y_surv_death["time"] = time_death

# Step 3. Replicate across imputations
n_imputations = len(imputations_list_jan26)
y_surv_readm_list = [y_surv_readm for _ in range(n_imputations)]
y_surv_death_list = [y_surv_death for _ in range(n_imputations)]

## PyCox

**We tuned a DeepHit competing-risks survival model for mortality and hospital readmission using five-fold stratified cross-validation and selected hyperparameters based on Uno’s inverse probability–weighted concordance index for mortality, treating readmission as a competing event.**

We implemented a stratified hyperparameter tuning procedure for a DeepHit competing-risks survival model, jointly modeling mortality and hospital readmission. Model training utilized 5-fold stratified cross-validation over the first imputed dataset. To ensure robust generalization across rare events and heterogeneous treatment modalities, stratified sampling was performed using composite labels representing the intersection of event type (mortality, readmission, or censoring) and care plan. Continuous covariates were standardized within training folds to prevent data leakage, and event times were discretized into 100 intervals using a data-driven cut-point transformation.

Rather than manual tuning, a systematic grid search was employed to identify the optimal configuration of **Hyperparameters**
- **Learning rate (LR)**: how fast the model updates its weights
- **Weight decay**: (reg) L2 regularization to reduce overfitting
- **Batch size**: Samples per training step (large batches help with rare events)
- **Dropout**: Fraction of neurons randomly dropped during training
- **Nodes**: Network depth and width
- **Alpha**: Strength of the ranking loss (time ordering)
- **Sigma**: Smoothing for the ranking loss
- **num_durations**: Fixed on 100 equidistant intervals (1 month per bin approx.)

A shared neural network architecture with cause-specific output heads was optimized over a grid of learning rates, weight decay regularization, network depth, and ranking-loss parameters (α,σ). Model discrimination was assessed using Uno’s Inverse Probability of Censoring Weighted (IPCW) C-index. To avoid overfitting to a single time point or outcome, the final hyperparameter configuration was selected based on the maximum composite C-index, averaged across both competing risks (mortality and readmission) and five annual evaluation horizons (1–5 years). All configurations and metrics were logged to ensure reproducibility.

In [19]:
#@title ⚡ Step 1: DeepHit Tuning (5-Fold, Thermal Safe, Multi-Horizon)
import itertools
import gc
import time
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchtuples as tt
import random
import os
from datetime import datetime
from pycox.models import DeepHit
from pycox.preprocessing.label_transforms import LabTransDiscreteTime
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import concordance_index_ipcw

# --- CONFIGURATION ---
NUM_RISKS = 3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Evaluation Horizons: We average performance over years 1-5 
# This is more robust than a single month, but cleaner than averaging all 108 months.
EVAL_HORIZONS = [12, 24, 36, 48, 60] 

# Suppress harmless PyTorch warnings
warnings.filterwarnings("ignore", message=".*weights_only=False.*")

# --- 0. REPRODUCIBILITY SEED ---
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# --- HELPER: Network Wrapper ---
class CauseSpecificNet(nn.Module):
    def __init__(self, in_f, nodes, out_f, dropout, num_risks):
        super().__init__()
        self.net = tt.practical.MLPVanilla(in_f, nodes, out_f, batch_norm=True, dropout=dropout)
        self.num_risks = num_risks
    def forward(self, x):
        return self.net(x).view(x.size(0), self.num_risks, -1)

# --- 1. DATA PREP ---
def prepare_stratified_data(df_idx=0):
    df = imputations_list_jan26[df_idx]
    y_d = y_surv_death_list[df_idx]
    y_r = y_surv_readm_list[df_idx]

    t_d = y_d['time'].values if hasattr(y_d['time'], 'values') else y_d['time']
    e_d_raw = y_d['event'].values if hasattr(y_d['event'], 'values') else y_d['event']
    e_r_raw = y_r['event'].values if hasattr(y_r['event'], 'values') else y_r['event']

    events = np.zeros(len(df), dtype=int)
    times = t_d.copy().astype('float32')
    e_d = e_d_raw.astype(bool)
    e_r = e_r_raw.astype(bool)

    # 🟢 Competing Risk Priority: Death (1) overrides Readm (2)
    events[e_r] = 2
    events[e_d] = 1 

    # Stratification Logic
    plan_cols = ['plan_type_corr_pg_pr', 'plan_type_corr_m_pr', 
                 'plan_type_corr_pg_pai', 'plan_type_corr_m_pai']
    available_plans = [c for c in plan_cols if c in df.columns]

    plan_category = np.zeros(len(df), dtype=int)
    for i, col in enumerate(available_plans, 1):
        plan_category[df[col] == 1] = i

    strat_labels = (events * 10) + plan_category
    return df, events, times, strat_labels

# --- 2. EXECUTION ---
X_all, events_all, times_all, strat_labels = prepare_stratified_data()
start_time = time.time()

# Updated Search Space
param_grid = {
    'lr': [1e-3, 1e-4], #(learning rate): Controls the step size of parameter updates during optimization; lower values promote more stable convergence, particularly important for rare-event survival outcomes.
    'weight_decay': [1e-4, 1e-3], #L2 regularization applied to network weights to reduce overfitting by penalizing large parameter values.
    'batch_size': [1024, 2048], #Number of observations processed per training step; larger batches improve gradient stability and event representation in datasets with low event incidence.
    'dropout': [0.2, 0.5], #Proportion of neurons randomly deactivated during training to prevent overfitting and improve generalization.
    'nodes': [[256, 256], [256, 256, 128]], #Defines the number and size of hidden layers in the neural network, controlling model capacity and representational complexity.
    'alpha': [0.2, 0.5], #Weight of the ranking loss component in DeepHit, regulating the emphasis on temporal risk discrimination across individuals.
    'sigma': [0.1, 0.5] #Smoothing parameter for the ranking loss that controls tolerance to small temporal ordering errors in event times.
}

keys, values = zip(*param_grid.items())
search_space = [dict(zip(keys, v)) for v in itertools.product(*values)]
tuning_results = []

print(f"⚡ Starting Defensible Tuning on {len(search_space)} combos...")

for i, params in enumerate(search_space):
    # 🟢 CHANGED: 5 Folds as requested
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_all, strat_labels)):
        torch.cuda.empty_cache()
        gc.collect()

        X_train, X_val = X_all.iloc[train_idx], X_all.iloc[val_idx]
        e_train, e_val = events_all[train_idx], events_all[val_idx]
        t_train, t_val = times_all[train_idx], times_all[val_idx]

        scaler = StandardScaler().fit(X_train)
        X_train_s = scaler.transform(X_train).astype('float32')
        X_val_s = scaler.transform(X_val).astype('float32')

        # 100 intervals balance resolution & stability
        labtrans = LabTransDiscreteTime(100)
        y_train = labtrans.fit_transform(t_train, e_train)
        y_val = labtrans.transform(t_val, e_val)
        y_train = (y_train[0].astype('int64'), y_train[1].astype('int64'))
        y_val = (y_val[0].astype('int64'), y_val[1].astype('int64'))

        in_f = X_train.shape[1]
        out_f = labtrans.out_features * NUM_RISKS

        net = CauseSpecificNet(in_f, params['nodes'], out_f, params['dropout'], NUM_RISKS)
        
        # Single Initialization (Corrected)
        model = DeepHit(net, tt.optim.Adam, 
                        alpha=params['alpha'], 
                        sigma=params['sigma'], 
                        duration_index=labtrans.cuts)        
        
        model.set_device(DEVICE)
        model.optimizer.set_lr(params['lr'])
        model.optimizer.param_groups[0]['weight_decay'] = params['weight_decay']

        try:
            model.fit(X_train_s, y_train, batch_size=params['batch_size'], epochs=50,
                      callbacks=[tt.callbacks.EarlyStopping()], verbose=False, val_data=(X_val_s, y_val))
            
            cif = model.predict_cif(X_val_s)
            
            # 🟢 UPDATED: Evaluate across 5 key years (12-60m) and average
            horizon_scores = []
            for h in EVAL_HORIZONS:
                idx_h = np.searchsorted(model.duration_index, h)
                if idx_h >= len(model.duration_index): idx_h = len(model.duration_index) - 1
                
                score_d = cif[1][idx_h, :] # Death
                score_r = cif[2][idx_h, :] # Readm

                y_tr_st = np.array([(bool(e==1), t) for e, t in zip(e_train, t_train)], dtype=[('e', bool), ('t', float)])
                y_va_st_d = np.array([(bool(e==1), t) for e, t in zip(e_val, t_val)], dtype=[('e', bool), ('t', float)])
                y_va_st_r = np.array([(bool(e==2), t) for e, t in zip(e_val, t_val)], dtype=[('e', bool), ('t', float)])

                c_d = concordance_index_ipcw(y_tr_st, y_va_st_d, score_d, tau=h)[0]
                c_r = concordance_index_ipcw(y_tr_st, y_va_st_r, score_r, tau=h)[0]
                horizon_scores.append((c_d + c_r) / 2)
            
            fold_scores.append(np.mean(horizon_scores))

        except Exception as e:
            # print(f"Fold Error: {e}") 
            fold_scores.append(np.nan)

        # Cleanup per fold
        del model; del net; gc.collect()

        # 🟢 THERMAL PAUSE
        # If the GPU temperature is high, enforce a short real cooldown period
        # This allows the cooling system to reduce core temperature and
        # helps prevent thermal throttling or unstable training behavior
        # (We run this after every fold to stay safe)
        print("❄️", end="") 
        time.sleep(30) 

    avg_s = np.nanmean(fold_scores)
    tuning_results.append({**params, 'score': avg_s})
    print(f"   [{i+1}/{len(search_space)}] Avg C-Index (1-5yr): {avg_s:.4f}")

# --- 3. RESULTS & EXPORT ---
results_df = pd.DataFrame(tuning_results).sort_values('score', ascending=False)
best_params = results_df.iloc[0].to_dict()

print("\n" + "="*60)
print(f"🏆 Best Config: {best_params}")
print("="*60)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
filename = f"DH_Tuning_5Fold_{timestamp}.csv"
results_df.to_csv(filename, index=False)
print(f"💾 Results saved to: {filename}")

elapsed_minutes = (time.time() - start_time) / 60
print(f"⏱️ Total Time: {elapsed_minutes:.2f} min")

⚡ Starting Defensible Tuning on 128 combos...
❄️❄️❄️❄️❄️   [1/128] Avg C-Index (1-5yr): 0.6977
❄️❄️❄️❄️❄️   [2/128] Avg C-Index (1-5yr): 0.6521
❄️❄️❄️❄️❄️   [3/128] Avg C-Index (1-5yr): 0.6722
❄️❄️❄️❄️❄️   [4/128] Avg C-Index (1-5yr): 0.6449
❄️❄️❄️❄️❄️   [5/128] Avg C-Index (1-5yr): 0.7061
❄️❄️❄️❄️❄️   [6/128] Avg C-Index (1-5yr): 0.6637
❄️❄️❄️❄️❄️   [7/128] Avg C-Index (1-5yr): 0.6920
❄️❄️❄️❄️❄️   [8/128] Avg C-Index (1-5yr): 0.6539
❄️❄️❄️❄️❄️   [9/128] Avg C-Index (1-5yr): 0.7183
❄️❄️❄️❄️❄️   [10/128] Avg C-Index (1-5yr): 0.7145
❄️❄️❄️❄️❄️   [11/128] Avg C-Index (1-5yr): 0.7111
❄️❄️❄️❄️❄️   [12/128] Avg C-Index (1-5yr): 0.7067
❄️❄️❄️❄️❄️   [13/128] Avg C-Index (1-5yr): 0.7281
❄️❄️❄️❄️❄️   [14/128] Avg C-Index (1-5yr): 0.7266
❄️❄️❄️❄️❄️   [15/128] Avg C-Index (1-5yr): 0.7264
❄️❄️❄️❄️❄️   [16/128] Avg C-Index (1-5yr): 0.7265
❄️❄️❄️❄️❄️   [17/128] Avg C-Index (1-5yr): 0.6943
❄️❄️❄️❄️❄️   [18/128] Avg C-Index (1-5yr): 0.6527
❄️❄️❄️❄️❄️   [19/128] Avg C-Index (1-5yr): 0.6734
❄️❄️❄️❄️❄️   

🏆 Best Config: {'lr': 0.001, 'weight_decay': 0.001, 'batch_size': 1024, 'dropout': 0.5, 'nodes': [256, 256, 128], 'alpha': 0.5, 'sigma': 0.5, 'score': 0.7371553906507226}

⏱️ Total Time: 823.12 min
Downloading "DH1_tun_par_20260203_1458.csv":

In [20]:
#@title 📝 Take-Home Message: Interpretation of Best DeepHit Configuration

import pandas as pd
from IPython.display import display

# --- HYPERPARAMETER INTERPRETATION DATAFRAME ---
config_interpretation = pd.DataFrame([
    {
        'Component': 'Regularization (The "Shield")',
        'Selected Value': 'Dropout: 0.5 | Weight Decay: 0.001 (High)',
        'Interpretation': 'The model required maximum "braking" power. The high dropout indicates the 4% mortality class is noisy; the model forces itself to ignore specific patient details (memorization) and learn only the most robust, universal predictors.'
    },
    {
        'Component': 'Model Capacity (Architecture)',
        'Selected Value': 'Nodes: [256, 256, 128] (Deep & Wide)',
        'Interpretation': 'Risk factors are highly non-linear. Simple "older = higher risk" logic is insufficient. The model requires deep layers to capture complex interactions, likely between Substance Type, Treatment Plan, and comorbidities.'
    },
    {
        'Component': 'Loss Function Strategy',
        'Selected Value': 'Alpha: 0.5 | Sigma: 0.5 (Balanced & Soft)',
        'Interpretation': 'The model prioritizes "Ranking" (who dies first) equally with "Timing" (when they die). The higher sigma (0.5) creates a "softer" margin of error, stabilizing the training against the high volume of censored data.'
    },
    {
        'Component': 'Optimization Mechanics',
        'Selected Value': 'Batch: 1024 | LR: 0.001 (Slow & Steady)',
        'Interpretation': 'Large batches are critical for rare events. A batch of 1024 ensures the model sees enough death cases in every single update to learn effectively, preventing "empty learning" steps common with smaller batches.'
    }
])

# --- DISPLAY ---
print("\n>>> TAKE-HOME MESSAGE: WHY THIS CONFIGURATION WON")
pd.set_option('display.max_colwidth', None)
display(config_interpretation)


>>> TAKE-HOME MESSAGE: WHY THIS CONFIGURATION WON


,Component,Selected Value,Interpretation
0,"Regularization (The ""Shield"")",Dropout: 0.5 | Weight Decay: 0.001 (High),"The model required maximum ""braking"" power. The high dropout indicates the 4% mortality class is noisy; the model forces itself to ignore specific patient details (memorization) and learn only the most robust, universal predictors."
1,Model Capacity (Architecture),"Nodes: [256, 256, 128] (Deep & Wide)","Risk factors are highly non-linear. Simple ""older = higher risk"" logic is insufficient. The model requires deep layers to capture complex interactions, likely between Substance Type, Treatment Plan, and comorbidities."
2,Loss Function Strategy,Alpha: 0.5 | Sigma: 0.5 (Balanced & Soft),"The model prioritizes ""Ranking"" (who dies first) equally with ""Timing"" (when they die). The higher sigma (0.5) creates a ""softer"" margin of error, stabilizing the training against the high volume of censored data."
3,Optimization Mechanics,Batch: 1024 | LR: 0.001 (Slow & Steady),"Large batches are critical for rare events. A batch of 1024 ensures the model sees enough death cases in every single update to learn effectively, preventing ""empty learning"" steps common with smaller batches."


Given that the initial hyperparameter sweep identified optimal values at the upper boundaries of the search space—specifically favoring maximum regularization (dropout 0.5, weight decay 0.001) and the deepest available network architecture—we conducted a **secondary, targeted 'zoom-in' grid search**. This follow-up analysis extended the search range to explore stronger regularization (dropout 0.6, weight decay 0.01) and increased model capacity (up to 512 nodes) to determine **if the performance plateau had truly been reached**. Parameters that demonstrated stability in the initial phase (batch size, alpha, and sigma) were fixed to their winning values, concentrating computational resources on fine-tuning the critical balance between model complexity and overfitting.

In [21]:
#@title ⚡ Step 1.5: Targeted "Zoom-In" Tuning (Pushing Boundaries)
import itertools
import gc
import time
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchtuples as tt
import random
import os
from datetime import datetime
from pycox.models import DeepHit
from pycox.preprocessing.label_transforms import LabTransDiscreteTime
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import concordance_index_ipcw

# --- CONFIGURATION ---
NUM_RISKS = 3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EVAL_HORIZONS = [12, 24, 36, 48, 60] 

warnings.filterwarnings("ignore", message=".*weights_only=False.*")

# --- REPRODUCIBILITY SEED ---
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# --- NETWORK WRAPPER ---
class CauseSpecificNet(nn.Module):
    def __init__(self, in_f, nodes, out_f, dropout, num_risks):
        super().__init__()
        self.net = tt.practical.MLPVanilla(in_f, nodes, out_f, batch_norm=True, dropout=dropout)
        self.num_risks = num_risks
    def forward(self, x):
        return self.net(x).view(x.size(0), self.num_risks, -1)

# --- DATA PREP ---
def prepare_stratified_data(df_idx=0):
    df = imputations_list_jan26[df_idx]
    y_d = y_surv_death_list[df_idx]
    y_r = y_surv_readm_list[df_idx]

    t_d = y_d['time'].values if hasattr(y_d['time'], 'values') else y_d['time']
    e_d_raw = y_d['event'].values if hasattr(y_d['event'], 'values') else y_d['event']
    e_r_raw = y_r['event'].values if hasattr(y_r['event'], 'values') else y_r['event']

    events = np.zeros(len(df), dtype=int)
    times = t_d.copy().astype('float32')
    e_d = e_d_raw.astype(bool)
    e_r = e_r_raw.astype(bool)

    events[e_r] = 2
    events[e_d] = 1 

    plan_cols = ['plan_type_corr_pg_pr', 'plan_type_corr_m_pr', 
                 'plan_type_corr_pg_pai', 'plan_type_corr_m_pai']
    available_plans = [c for c in plan_cols if c in df.columns]

    plan_category = np.zeros(len(df), dtype=int)
    for i, col in enumerate(available_plans, 1):
        plan_category[df[col] == 1] = i

    strat_labels = (events * 10) + plan_category
    return df, events, times, strat_labels

# --- EXECUTION ---
X_all, events_all, times_all, strat_labels = prepare_stratified_data()
start_time = time.time()

# 🚀 TARGETED SEARCH SPACE (Based on Previous Winners)
param_grid = {
    # Test slightly higher LR vs current winner (0.001)
    'lr': [0.001, 0.003], 
    
    # Test stronger regularization vs current winner (0.001)
    'weight_decay': [0.001, 0.01],
    
    # Test extreme dropout vs current winner (0.5)
    'dropout': [0.5, 0.6], 
    
    # Test wider capacity vs current winner ([256, 256, 128])
    'nodes': [[256, 256, 128], [512, 256, 128]], 
    
    # Fixed best performers from previous run to save time
    'batch_size': [1024], 
    'alpha': [0.5], 
    'sigma': [0.5] 
}

keys, values = zip(*param_grid.items())
search_space = [dict(zip(keys, v)) for v in itertools.product(*values)]
tuning_results = []

print(f"⚡ Starting Targeted 'Zoom-In' Tuning on {len(search_space)} combos...")

for i, params in enumerate(search_space):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_all, strat_labels)):
        torch.cuda.empty_cache()
        gc.collect()

        X_train, X_val = X_all.iloc[train_idx], X_all.iloc[val_idx]
        e_train, e_val = events_all[train_idx], events_all[val_idx]
        t_train, t_val = times_all[train_idx], times_all[val_idx]

        scaler = StandardScaler().fit(X_train)
        X_train_s = scaler.transform(X_train).astype('float32')
        X_val_s = scaler.transform(X_val).astype('float32')

        labtrans = LabTransDiscreteTime(100)
        y_train = labtrans.fit_transform(t_train, e_train)
        y_val = labtrans.transform(t_val, e_val)
        y_train = (y_train[0].astype('int64'), y_train[1].astype('int64'))
        y_val = (y_val[0].astype('int64'), y_val[1].astype('int64'))

        in_f = X_train.shape[1]
        out_f = labtrans.out_features * NUM_RISKS

        net = CauseSpecificNet(in_f, params['nodes'], out_f, params['dropout'], NUM_RISKS)
        model = DeepHit(net, tt.optim.Adam, alpha=params['alpha'], sigma=params['sigma'], duration_index=labtrans.cuts)        
        model.set_device(DEVICE)
        model.optimizer.set_lr(params['lr'])
        model.optimizer.param_groups[0]['weight_decay'] = params['weight_decay']

        try:
            model.fit(X_train_s, y_train, batch_size=params['batch_size'], epochs=50,
                      callbacks=[tt.callbacks.EarlyStopping()], verbose=False, val_data=(X_val_s, y_val))
            
            cif = model.predict_cif(X_val_s)
            
            # Evaluate across 5 key years (12-60m) and average
            horizon_scores = []
            for h in EVAL_HORIZONS:
                idx_h = np.searchsorted(model.duration_index, h)
                if idx_h >= len(model.duration_index): idx_h = len(model.duration_index) - 1
                
                score_d = cif[1][idx_h, :] # Death
                score_r = cif[2][idx_h, :] # Readm

                y_tr_st = np.array([(bool(e==1), t) for e, t in zip(e_train, t_train)], dtype=[('e', bool), ('t', float)])
                y_va_st_d = np.array([(bool(e==1), t) for e, t in zip(e_val, t_val)], dtype=[('e', bool), ('t', float)])
                y_va_st_r = np.array([(bool(e==2), t) for e, t in zip(e_val, t_val)], dtype=[('e', bool), ('t', float)])

                c_d = concordance_index_ipcw(y_tr_st, y_va_st_d, score_d, tau=h)[0]
                c_r = concordance_index_ipcw(y_tr_st, y_va_st_r, score_r, tau=h)[0]
                horizon_scores.append((c_d + c_r) / 2)
            
            fold_scores.append(np.mean(horizon_scores))

        except Exception as e:
            fold_scores.append(np.nan)

        del model; del net; gc.collect()
        
        # Short pause to prevent thermal throttling
        time.sleep(5) 

    avg_s = np.nanmean(fold_scores)
    tuning_results.append({**params, 'score': avg_s})
    print(f"   [{i+1}/{len(search_space)}] Avg C-Index (1-5yr): {avg_s:.4f}")

# --- RESULTS ---
results_df = pd.DataFrame(tuning_results).sort_values('score', ascending=False)
best_params = results_df.iloc[0].to_dict()

print("\n" + "="*60)
print(f"🏆 Best Zoom-In Config: {best_params}")
print("="*60)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
filename = f"DH_ZoomIn_Tuning_{timestamp}.csv"
results_df.to_csv(filename, index=False)
print(f"💾 Saved to: {filename}")

⚡ Starting Targeted 'Zoom-In' Tuning on 16 combos...
   [1/16] Avg C-Index (1-5yr): 0.7373
   [2/16] Avg C-Index (1-5yr): 0.7331
   [3/16] Avg C-Index (1-5yr): 0.7321
   [4/16] Avg C-Index (1-5yr): 0.7316
   [5/16] Avg C-Index (1-5yr): 0.7008
   [6/16] Avg C-Index (1-5yr): 0.6936
   [7/16] Avg C-Index (1-5yr): 0.6946
   [8/16] Avg C-Index (1-5yr): 0.6987
   [9/16] Avg C-Index (1-5yr): 0.7145
   [10/16] Avg C-Index (1-5yr): 0.7191
   [11/16] Avg C-Index (1-5yr): 0.7165
   [12/16] Avg C-Index (1-5yr): 0.7106
   [13/16] Avg C-Index (1-5yr): 0.6884
   [14/16] Avg C-Index (1-5yr): 0.6802
   [15/16] Avg C-Index (1-5yr): 0.6863
   [16/16] Avg C-Index (1-5yr): 0.6813

🏆 Best Zoom-In Config: {'lr': 0.001, 'weight_decay': 0.001, 'dropout': 0.5, 'nodes': [256, 256, 128], 'batch_size': 1024, 'alpha': 0.5, 'sigma': 0.5, 'score': 0.7372626094515411}
💾 Saved to: DH_ZoomIn_Tuning_20260208_1440.csv


To identify the optimal hyperparameter configuration, we conducted a two-stage tuning process using 5-fold stratified cross-validation on the first imputed dataset. An initial coarse grid search explored a broad range of learning rates, regularization strengths (weight decay, dropout), and network architectures. The top-performing configuration from this phase (C-index: 0.737) favored high model capacity and strong regularization, residing at the upper boundaries of the search space. Consequently, a secondary, targeted 'zoom-in' search was performed to explore even strictly higher regularization and deeper architectures. This confirmatory step yielded negligible performance gains (C-index: 0.7373 vs. 0.7371), indicating that the model had reached its convergence plateau. The final selected configuration (Learning Rate: 0.001, Weight Decay: 0.001, Dropout: 0.5, Nodes: [256, 256, 128]) was thus confirmed as both robust and optimal for the subsequent 10-fold evaluation.